In [11]:
import importlib
import load_data_BCICIV

importlib.reload(load_data_BCICIV)
from load_data_BCICIV import load_all_subjects

data_path = 'datasets/BCICIV_2a_gdf'

data = load_all_subjects(data_path)

/home/alumno/.local/share/uv/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: datasets/BCICIV_2a_gdf/A01T.gdf
A01: 144 loaded trials


/home/alumno/.local/share/uv/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: datasets/BCICIV_2a_gdf/A02T.gdf
A02: 144 loaded trials


/home/alumno/.local/share/uv/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

Loaded file: datasets/BCICIV_2a_gdf/A07T.gdf
A07: 144 loaded trials


/home/alumno/.local/share/uv/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: datasets/BCICIV_2a_gdf/A08T.gdf
A08: 144 loaded trials


/home/alumno/.local/share/uv/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: datasets/BCICIV_2a_gdf/A09T.gdf
A09: 144 loaded trials


In [12]:
print(data["X"].shape)
print(data["y"].shape)
print(data["subject_ids"])

(1296, 875, 22)
(1296,)
['A01' 'A01' 'A01' ... 'A09' 'A09' 'A09']


In [13]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, GroupKFold
import numpy as np
import gc

## INITIAL MODELS FOR BCICIV

In [14]:
X = data['X']
y = data['y']
subjects = data['subject_ids']
unique_subjects = np.unique(subjects)

In [15]:
transpose = FunctionTransformer(lambda X: np.transpose(X, (0, 2, 1)), validate=False)

pipe1 = Pipeline([
    ('transpose', transpose),
    ('csp', CSP(n_components=4, reg=None, log=True, norm_trace=False)),
    ('clf', LinearDiscriminantAnalysis())
])

pipe2 = Pipeline([
    ('transpose', transpose),
    ('csp', CSP(n_components=4, reg=None, log=True, norm_trace=False)),
    ('clf', SVC(kernel='rbf', C=1.0, gamma='scale'))
])

In [16]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe1.fit(X_train, y_train)

accuracy = pipe1.score(X_train, y_train)
print(f"Training Accuracy: {accuracy:.4f}")
accuracy = pipe1.score(X_val, y_val)
print(f"Validation Accuracy: {accuracy:.4f}")


pipe2.fit(X_train, y_train)

accuracy = pipe2.score(X_train, y_train)
print(f"Training Accuracy: {accuracy:.4f}")
accuracy = pipe2.score(X_val, y_val)
print(f"Validation Accuracy: {accuracy:.4f}")

Computing rank from data with rank=None
    Using tolerance 0.00021 (2.2e-16 eps * 22 dim * 4.3e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Training Accuracy: 0.5512
Validation Accuracy: 0.5269
Computing rank from data with rank=None
    Using tolerance 0.00021 (2.2e-16 eps * 22 dim * 4.3e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Training Accuracy: 0.6149
Validation Accuracy: 0.5731


## LEAVE ONE GROUP OUT CROSS VALIDATION

In [17]:
from sklearn.model_selection import LeaveOneGroupOut 
from sklearn.base import clone 

def run_loso(X, y, subj, pipe):

    fold_accuracies = [] 
    loso = LeaveOneGroupOut()

    for i, (train_idx, val_idx) in enumerate(loso.split(X,y,subj)):
        print(f"\t Fold {i}")

        X_train = X[train_idx]
        y_train = y[train_idx]

        X_val = X[val_idx]
        y_val = y[val_idx]

        model = clone(pipe)
        model.fit(X_train, y_train)

        acc = model.score(X_val, y_val)
        print(f"Val acc: {acc}")
        fold_accuracies.append(acc)

        del X_train, y_train, X_val, y_val
        gc.collect()

    mean_acc = np.mean(np.array(fold_accuracies))

    final_model = clone(pipe)
    final_model.fit(X, y)

    return final_model, mean_acc, fold_accuracies



In [18]:
final_model, mean_acc, fold_accuracies = run_loso(X, y, subjects, pipe1)

	 Fold 0
Computing rank from data with rank=None
    Using tolerance 0.00022 (2.2e-16 eps * 22 dim * 4.5e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Val acc: 0.5763888888888888
	 Fold 1
Computing rank from data with rank=None
    Using tolerance 0.00022 (2.2e-16 eps * 22 dim * 4.6e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Val acc: 0.5416666666666666
	 Fold 2
Computing rank from data with rank=None
    Using tolerance 0.00022 (2.2e-16 eps * 22 dim * 4.4e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 

In [19]:
mean_acc

0.5540123456790123